# AHR predictive learning v3: shared definitions and provenance

This versioned experiment preserves the completed notebooks 00–13 and their results. It tests consistent Queen-cycle budgets, broader exact AHR search, verified tactical training, persistent trajectories and action-conditioned latent prediction. External Go engines supply evaluation only. All new executable code remains in notebooks.

**Inputs:** original tagged definitions in notebooks 00–06 and 08–11; the verified selected baseline. **Outputs:** definition-only imports, source fingerprints and an experiment readiness receipt. **Expected runtime:** under one minute. **Training:** none. **Recovery:** fresh `ahr-go` kernel, Run All; resumable scientific state belongs to the training/evaluation/controller run directories named by their notebooks. Off-VM copies remain deferred by the user.

AHR's learned rankings and latent predictions do not certify strategic outcomes. Exact state transitions and the independent verifier remain authoritative for legality; cutoffs stay UNKNOWN.

In [ ]:
from pathlib import Path
import copy, hashlib, json, os, time, uuid
import nbformat

V3_ROOT='experiments/ahr_jepa_v3'
V3_SCHEMA='ahr_predictive_go_v3'
V3_BASELINE='go09-first-selfplay-v1'
V3_BASELINE_CHECKPOINT='3e4287bc1e8885fa975660b07bc76800e22aa6f8627fd896cc3163280b1f95ad'
V3_NOTEBOOKS=('00_shared.ipynb','08_predictive_ahr.ipynb','09_trajectory_learning.ipynb','11_controlled_evaluation.ipynb')

def v3_project(start=None):
    project=Path.cwd().resolve() if start is None else Path(start).resolve()
    while not (project/'notebooks/00_instance_setup_and_recovery.ipynb').is_file():
        if project.parent==project:raise RuntimeError('Run inside the AHR Go workspace')
        project=project.parent
    return project

def v3_import_exports(path, namespace):
    path=Path(path)
    book=nbformat.read(path,4);nbformat.validate(book)
    count=0
    for cell in book.cells:
        if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):
            exec(compile(cell.source,str(path)+':ahr-export','exec'),namespace)
            count+=1
    if not count:raise RuntimeError('No tagged definitions in '+str(path))
    return count

def v3_load_upstream(project, namespace):
    project=Path(project)
    for number in (*range(7),8,9,10,11):
        paths=list((project/'notebooks').glob(f'{number:02d}_*.ipynb'))
        if len(paths)!=1:raise RuntimeError('Ambiguous upstream notebook '+str(number))
        v3_import_exports(paths[0],namespace)
    namespace['torch'].set_num_threads(1)
    return namespace

def v3_code_sha(path, exports_only=False):
    book=nbformat.read(path,4);nbformat.validate(book)
    source=[cell.source for cell in book.cells if cell.cell_type=='code' and
            (not exports_only or 'ahr-export' in cell.metadata.get('tags',[]))]
    return hashlib.sha256(json.dumps(source,ensure_ascii=False).encode()).hexdigest()

def v3_source_identity(project, require_all=True):
    project=Path(project)
    paths=sorted((project/'notebooks').glob('*.ipynb'))
    for name in V3_NOTEBOOKS:
        path=project/V3_ROOT/name
        if path.exists():paths.append(path)
        elif require_all:raise FileNotFoundError(path)
    return {str(path.relative_to(project)):v3_code_sha(path,True) for path in paths}

def v3_digest(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()

def v3_read_json(path):
    return json.loads(Path(path).read_text())

def v3_report_path(project,name):
    if Path(name).name!=name:raise ValueError('Simple report name required')
    directory=Path(project)/'reports/ahr_jepa_v3';directory.mkdir(parents=True,exist_ok=True)
    return directory/name

def v3_seed(seed):
    import random,numpy as np,torch
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True
    torch.backends.cuda.matmul.allow_tf32=False
    torch.backends.cudnn.allow_tf32=False
    random.seed(seed);np.random.seed(seed%(2**32));torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

def v3_verify_original_sources(project):
    project=Path(project)
    frozen=v3_read_json(project/'config/go_generations_v2.json')
    for relative,expected in frozen['sources'].items():
        # v2 source contract serializes all code cells with default ASCII behavior.
        book=nbformat.read(project/relative,4)
        actual=hashlib.sha256(json.dumps([c.source for c in book.cells if c.cell_type=='code']).encode()).hexdigest()
        if actual!=expected:raise RuntimeError('Historical source changed: '+relative)
    return frozen['configuration_sha256']


In [ ]:
PROJECT=v3_project()
old_configuration=v3_verify_original_sources(PROJECT)
v3_load_upstream(PROJECT,globals())
baseline,metadata=load_go_ahr(PROJECT,V3_BASELINE,'cpu','latest')
assert metadata['checkpoint_sha256']==V3_BASELINE_CHECKPOINT
receipt={'schema':V3_SCHEMA,'status':'PASS','training_steps':0,
         'original_configuration_sha256':old_configuration,
         'baseline_checkpoint_sha256':metadata['checkpoint_sha256'],
         'source_count':len(v3_source_identity(PROJECT,require_all=False)),
         'imports':'ahr-export definitions only','off_instance_backup':'DEFERRED_BY_USER'}
atomic_json(v3_report_path(PROJECT,'shared_readiness.json'),receipt)
print(json.dumps(receipt,indent=2))